# Run SCEPTER Scenarios

This notebook is the execution step after `04_test_scepter_inputs.ipynb`. It reads the staged SCEPTER run table, writes one run folder/config file per model-unit/scenario combination, and optionally calls an external SCEPTER command.

The repo does not yet define the exact local SCEPTER executable or template format, so execution is controlled by `RUN_EXTERNAL_SCEPTER`. Leave it as `False` to stage configs and inspect them. Set it to `True` only after `SCEPTER_COMMAND_TEMPLATE` points to a working local SCEPTER command.


## Workflow

1. **Load staged run table** from `data/scepter_runs/inputs/scepter_runs.csv`.
2. **Select test runs** so we can start small before launching a full batch.
3. **Write run configs** under `data/scepter_runs/runs/{run_id}/scepter_config.json`.
4. **Create output folders** under `data/scepter_runs/data/outputs/{run_id}/`.
5. **Optionally execute SCEPTER** using a command template that receives `{config_path}`, `{run_dir}`, `{output_dir}`, and `{run_id}`.
6. **Write an execution manifest** to `data/scepter_runs/data/outputs/scepter_execution_manifest.csv`.

The execution manifest is the handoff to the next notebook, which will parse completed SCEPTER outputs and join them back to parcel/model-unit geometries.


In [ ]:
from pathlib import Path
import os
import sys

import pandas as pd

def mount_google_drive_if_colab() -> None:
    try:
        from google.colab import drive
    except ModuleNotFoundError:
        return

    drive.mount("/content/drive")


mount_google_drive_if_colab()

COLAB_PROJECT_ROOT = Path("/content/drive/MyDrive/erw_spatial_mrv")
COLAB_DATA_ROOT = COLAB_PROJECT_ROOT / "data"
LOCAL_PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()


def has_erw_package(project_root: Path) -> bool:
    return (project_root / "src" / "erw_mrv" / "__init__.py").exists()


def source_root_candidates() -> list[Path]:
    cwd = Path.cwd().resolve()
    candidates = [LOCAL_PROJECT_ROOT, COLAB_PROJECT_ROOT]
    for base in (cwd, *cwd.parents):
        candidates.extend((base, base / "erw_spatial_mrv"))
    candidates.extend(
        Path(path)
        for path in (
            "/content/erw_spatial_mrv",
            "/content/enhanced_rock_weathering/erw_spatial_mrv",
            "/content/drive/MyDrive/erw_spatial_mrv",
        )
    )
    unique = []
    for candidate in candidates:
        if candidate not in unique:
            unique.append(candidate)
    return unique


def find_source_project_root() -> Path:
    for candidate in source_root_candidates():
        if has_erw_package(candidate):
            return candidate
    checked = chr(10).join(f"- {candidate}" for candidate in source_root_candidates())
    raise ModuleNotFoundError(
        "Could not find src/erw_mrv. The data can live in Google Drive, but "
        "the notebook still needs the project source folder containing src/erw_mrv. "
        "In Colab, upload/sync the full erw_spatial_mrv project or run from a "
        "checkout that includes src/. Checked: "
        f"{checked}"
    )


SOURCE_PROJECT_ROOT = find_source_project_root()
PROJECT_ROOT = COLAB_PROJECT_ROOT if COLAB_DATA_ROOT.exists() else SOURCE_PROJECT_ROOT
DATA_ROOT = COLAB_DATA_ROOT if COLAB_DATA_ROOT.exists() else PROJECT_ROOT / "data"
os.environ["ERW_MRV_DATA_ROOT"] = str(DATA_ROOT)

SRC = SOURCE_PROJECT_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

print(f"PROJECT_ROOT = {PROJECT_ROOT}")
print(f"SOURCE_PROJECT_ROOT = {SOURCE_PROJECT_ROOT}")
print(f"DATA_ROOT = {DATA_ROOT}")

from erw_mrv.paths import SCEPTER_INPUTS, SCEPTER_OUTPUTS, SCEPTER_RUN_DIR, ensure_dir
from erw_mrv.scepter import (
    execute_scepter_runs,
    load_scepter_runs,
    select_scepter_runs,
    write_run_configs,
)

pd.set_option("display.max_columns", 80)
pd.set_option("display.max_colwidth", 160)


## Settings

Start with a small number of runs. Once the command works and outputs look right, set `MAX_TEST_RUNS = None` to stage or execute every row.


In [ ]:
RUN_TABLE_PATH = SCEPTER_INPUTS / "scepter_runs.csv"
RUN_ROOT = ensure_dir(SCEPTER_RUN_DIR)
OUTPUT_ROOT = ensure_dir(SCEPTER_OUTPUTS)

MAX_TEST_RUNS = 12
SCENARIO_IDS = None  # for example: ["baseline_no_erw", "erw_20t_fine"]

RUN_EXTERNAL_SCEPTER = False
SCEPTER_COMMAND_TEMPLATE = "scepter --config {config_path} --output-dir {output_dir}"
TIMEOUT_SECONDS = 60 * 60

MANIFEST_PATH = OUTPUT_ROOT / "scepter_execution_manifest.csv"
RUN_ROOT, OUTPUT_ROOT, MANIFEST_PATH


## Load Run Table

This table should come from notebook `04`. Each row is one model unit under one ERW scenario.


In [ ]:
runs = load_scepter_runs(RUN_TABLE_PATH)
print(f"Available staged runs: {len(runs):,}")
runs.groupby("scenario_id").size().rename("run_count").reset_index()


## Select Runs For This Batch

This keeps the first execution batch manageable. The default `MAX_TEST_RUNS = 12` is enough to test all four scenarios across a few model units.


In [ ]:
selected_runs = select_scepter_runs(
    runs,
    max_runs=MAX_TEST_RUNS,
    scenario_ids=SCENARIO_IDS,
)

print(f"Runs selected for this batch: {len(selected_runs):,}")
selected_runs[["run_id", "model_unit_id", "scenario_id", "area_ha", "basalt_application_t_ha", "basalt_d50_um"]].head(20)


## Stage Run Configs

Each run gets a folder under `data/scepter_runs/runs/` and a JSON configuration file. The config is deliberately simple and auditable; later we can replace the JSON writer with the exact SCEPTER input-file format if needed.


In [ ]:
execution_table = write_run_configs(
    selected_runs,
    run_root=RUN_ROOT,
    output_root=OUTPUT_ROOT,
)

execution_table.head()


## Inspect One Config

Before running an external model, inspect one generated config to confirm units and scenario parameters look reasonable.


In [ ]:
example_config = Path(execution_table.iloc[0]["config_path"])
print(example_config)
print(example_config.read_text()[:2000])


## Execute Or Stage Manifest

When `RUN_EXTERNAL_SCEPTER = False`, this cell writes a manifest showing that runs were staged but not executed. When set to `True`, it calls the command template once per run.

The command template can use these fields: `{run_id}`, `{run_dir}`, `{output_dir}`, `{config_path}`, `{model_unit_id}`, and `{scenario_id}`.


In [ ]:
if RUN_EXTERNAL_SCEPTER:
    manifest = execute_scepter_runs(
        execution_table,
        command_template=SCEPTER_COMMAND_TEMPLATE,
        timeout_seconds=TIMEOUT_SECONDS,
    )
else:
    manifest = execution_table.copy()
    manifest["command"] = SCEPTER_COMMAND_TEMPLATE
    manifest["status"] = "staged_not_executed"
    manifest["return_code"] = pd.NA
    manifest["elapsed_seconds"] = 0.0
    manifest["stdout_log"] = pd.NA
    manifest["stderr_log"] = pd.NA

manifest.to_csv(MANIFEST_PATH, index=False)
manifest[["run_id", "scenario_id", "status", "config_path", "output_dir"]].head(20)


## Batch Summary


In [ ]:
status_summary = manifest.groupby("status").size().rename("run_count").reset_index()
scenario_summary = manifest.groupby(["scenario_id", "status"]).size().rename("run_count").reset_index()

status_summary, scenario_summary, MANIFEST_PATH


## Expected SCEPTER Outputs

Once external execution is enabled, each run folder under `data/scepter_runs/data/outputs/{run_id}/` should contain whatever files SCEPTER produces for that scenario. The next notebook will need to parse those outputs into a clean table with at least:

- `run_id`
- `model_unit_id`
- `scenario_id`
- weathering or alkalinity metric
- cumulative CO2 removal
- CO2 removal per hectare
- soil pH or chemistry changes if available

The parser will use `scepter_execution_manifest.csv` to know which runs completed and where their output folders are.


## Outputs From This Notebook

This notebook writes SCEPTER batch execution artifacts:

- `data/scepter_runs/runs/{run_id}/scepter_config.json`: one model config per selected run.
- `data/scepter_runs/data/outputs/{run_id}/`: output folder reserved for each selected run.
- `data/scepter_runs/data/outputs/scepter_execution_manifest.csv`: status table for staged or executed runs.
- `data/scepter_runs/runs/{run_id}/scepter_stdout.log`: stdout log, written only when `RUN_EXTERNAL_SCEPTER = True`.
- `data/scepter_runs/runs/{run_id}/scepter_stderr.log`: stderr log, written only when `RUN_EXTERNAL_SCEPTER = True`.

The next step is notebook `06`, which should read the execution manifest, parse completed SCEPTER output files, and join model results back to spatial model units.
